# Solar Data Feature Engineering and Preprocessing (v2)

This notebook loads `data/PV_Data.csv` and performs a full preprocessing and feature engineering pipeline for solar power modeling.

## 1) Imports and Display Settings

This setup cell imports all required libraries for preprocessing, feature selection, and scaling. It also sets pandas display options to make wide tables easier to inspect during analysis.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


## 2) Load and Validate Raw Dataset

This cell reads `data/PV_Data.csv`, standardizes column names to uppercase, and verifies that required fields (`DATE` and `POWER`) are present.

A preview is shown to confirm schema and data format.


In [2]:
# Load raw data
data_path = Path("data") / "PV_Data.csv"
df = pd.read_csv(data_path)
df.columns = [c.strip().upper() for c in df.columns]

if "DATE" not in df.columns:
    raise ValueError("Expected a DATE column in data/PV_Data.csv")
if "POWER" not in df.columns:
    raise ValueError("Expected a POWER column in data/PV_Data.csv")

print("Raw shape:", df.shape)
display(df.head())


Raw shape: (19705, 16)


,INDEX,DATE,TIME_INDEX,TCWL,TCIW,SP,HUM,TCC,U,V,TEMP,TP,SSRD,STRD,TSR,POWER
0,0.0,4/1/2012 0:00,1.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.754103
1,1.0,4/1/2012 1:00,2.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.555000
2,2.0,4/1/2012 2:00,3.0,0.005524,0.033575,94757.9375,54.678604,0.457138,2.482865,-2.993330,295.651367,0.000000,2671979.0,1239402.0,3038689.5,0.438397
3,3.0,4/1/2012 3:00,4.0,0.030113,0.132009,94732.8125,61.294891,0.771429,3.339867,-1.982535,294.454590,0.001341,2252213.5,1237373.5,2691150.5,0.145449
4,4.0,4/1/2012 4:00,5.0,0.057167,0.110645,94704.0625,67.775284,0.965866,3.106102,-1.446051,293.261475,0.002501,1610654.5,1286522.0,2083191.0,0.111987


In [3]:
df.describe()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19705 entries, 0 to 19704
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   INDEX       8834 non-null   float64
 1   DATE        19705 non-null  str    
 2   TIME_INDEX  8834 non-null   float64
 3   TCWL        19705 non-null  float64
 4   TCIW        19705 non-null  float64
 5   SP          19705 non-null  float64
 6   HUM         19705 non-null  float64
 7   TCC         19705 non-null  float64
 8   U           19705 non-null  float64
 9   V           19705 non-null  float64
 10  TEMP        19705 non-null  float64
 11  TP          19705 non-null  float64
 12  SSRD        19705 non-null  float64
 13  STRD        19705 non-null  float64
 14  TSR         19705 non-null  float64
 15  POWER       19704 non-null  float64
dtypes: float64(15), str(1)
memory usage: 2.4 MB


## 3) Missing Data Profile and Row-Level Quality Filter

This cell summarizes missing values by feature and removes rows with excessive missingness (more than 50% missing fields).

It provides a quick quality baseline before time alignment and imputation.


In [4]:
# 1) Data quality check + missing value profiling
missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})

print("Missing values by column:")
display(missing_report[missing_report["missing_count"] > 0])

# Remove rows!!!! with excessive missingness (more than 50%)
row_missing_ratio = df.isna().mean(axis=1)
drop_mask = row_missing_ratio > 0.50
print(f"Rows dropped for excessive missingness: {drop_mask.sum()}")
df = df.loc[~drop_mask].copy()


Missing values by column:


,missing_count,missing_pct
INDEX,10871,55.168739
TIME_INDEX,10871,55.168739
POWER,1,0.005075


Rows dropped for excessive missingness: 0


In [5]:
# # df.isna().mean().sort_values(ascending=False)
# row_missing_ratio = df.isna().mean(axis=1)
# row_missing_ratio 
print(df.index)

RangeIndex(start=0, stop=19705, step=1)


## 4) Timestamp Standardization and Integrity Checks

The `DATE` field is parsed into a proper datetime index, invalid timestamps are removed, and records are sorted in chronological order.

The notebook also removes duplicate timestamps, reindexes to a complete hourly timeline, and validates expected daily counts (24 observations/day).


In [6]:
# 1. Check DATE column exists
if "DATE" not in df.columns:
    raise ValueError("DATE column is missing.")

# 2. Convert DATE to datetime and remove invalid dates
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
df = df.dropna(subset=["DATE"])

# 3. Sort chronologically
df = df.sort_values("DATE").reset_index(drop=True)

# 4. Ensure data starts at 00:00
first_midnight_rows = df[df["DATE"].dt.hour == 0]

# Find the first timestamp with hour 00 and remove all previous rows,
# and then reindex the dataframe
if len(first_midnight_rows) > 0:
    first_midnight_idx = first_midnight_rows.index[0]
    df = df.loc[first_midnight_idx:].reset_index(drop=True)
else:
    raise ValueError("No timestamp with hour 00 found in the dataset.")

# 5. Count records per day
daily_counts = df.groupby(df["DATE"].dt.date).size()

print("Daily record counts:")
print(daily_counts.head())

# 6. Remove days that do not have exactly 24 hourly observations
valid_days = daily_counts[daily_counts == 24].index
df = df[df["DATE"].dt.date.isin(valid_days)].reset_index(drop=True)

# 7. Final summary
print(f"Number of valid days: {len(valid_days)}")
print(f"Number of records: {len(df)}")

# Optional: verify all remaining days have 24 records
print("\nVerification:")
print(df.groupby(df["DATE"].dt.date).size().value_counts())

Daily record counts:
DATE
2012-04-01    24
2012-04-02    24
2012-04-03    24
2012-04-04    24
2012-04-05    24
dtype: int64
Number of valid days: 821
Number of records: 19704

Verification:
24    821
Name: count, dtype: int64


In [7]:
# first_midnight_rows
# df.head()
df

,INDEX,DATE,TIME_INDEX,TCWL,TCIW,SP,HUM,TCC,U,V,TEMP,TP,SSRD,STRD,TSR,POWER
0,0.0,2012-04-01 00:00:00,1.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.754103
1,1.0,2012-04-01 01:00:00,2.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.555000
2,2.0,2012-04-01 02:00:00,3.0,0.005524,0.033575,94757.9375,54.678604,0.457138,2.482865,-2.993330,295.651367,0.000000,2671979.0,1239402.0,3038689.5,0.438397
3,3.0,2012-04-01 03:00:00,4.0,0.030113,0.132009,94732.8125,61.294891,0.771429,3.339867,-1.982535,294.454590,0.001341,2252213.5,1237373.5,2691150.5,0.145449
4,4.0,2012-04-01 04:00:00,5.0,0.057167,0.110645,94704.0625,67.775284,0.965866,3.106102,-1.446051,293.261475,0.002501,1610654.5,1286522.0,2083191.0,0.111987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19699,NaN,2014-06-30 19:00:00,NaN,0.001213,0.000000,95132.1250,85.635208,0.055908,2.523030,0.077032,276.883301,0.000000,0.0,946376.0,0.0,0.000000
19700,NaN,2014-06-30 20:00:00,NaN,0.000668,0.000000,95168.9375,88.063522,0.030334,2.378217,-0.451427,276.305176,0.000000,0.0,927472.0,0.0,0.000064
19701,NaN,2014-06-30 21:00:00,NaN,0.000488,0.000000,95211.0000,90.338135,0.008423,2.118501,-0.930668,275.912353,0.000000,40287.5,920304.0,68573.5,0.013846
19702,NaN,2014-06-30 22:00:00,NaN,0.000877,0.000000,95257.4375,90.855988,0.116913,2.239975,-1.051635,276.846680,0.000000,326096.0,933800.0,427707.5,0.043718


## 5) Physical Validation and Missing Value Imputation

This step enforces realistic physical bounds (for example humidity between 0 and 100, non-negative `POWER`, `SSRD`, and `TP`).

Missingness indicator columns are also added for key variables, and numeric gaps are imputed using time-based interpolation followed by forward/backward fill.


In [8]:
# 3) Validity checks and imputation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# # Ensure DatetimeIndex for time-weighted interpolation
# if not isinstance(df.index, pd.DatetimeIndex):
#     if "DATE" in df.columns:
#         print("df.index, pd.DatetimeIndex")
#         date_idx = pd.to_datetime(df["DATE"], errors="coerce")
#         df = df.loc[date_idx.notna()].copy()
#         df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
#         df = df.set_index("DATE", drop=False).sort_index()
#     else:
#         raise ValueError("Time interpolation requires DatetimeIndex or DATE column.")

# Physical validity checks where applicable
if "HUM" in df.columns:
    df.loc[(df["HUM"] < 0) | (df["HUM"] > 100), "HUM"] = np.nan
if "POWER" in df.columns:
    df.loc[df["POWER"] < 0, "POWER"] = np.nan
if "SSRD" in df.columns:
    df.loc[df["SSRD"] < 0, "SSRD"] = np.nan
if "TP" in df.columns:
    df.loc[df["TP"] < 0, "TP"] = np.nan

# Missingness indicator features for key sensors
for col in ["HUM", "TEMP", "SSRD", "POWER"]:
    if col in df.columns:
        df[f"MISS_{col}"] = df[col].isna().astype(int)

# Time-aware imputation for numeric columns
try:
    df[numeric_cols] = df[numeric_cols].interpolate(method="time", limit_direction="both")
except ValueError:
    # Fallback for edge cases where index unexpectedly loses datetime semantics
    df[numeric_cols] = df[numeric_cols].interpolate(method="linear", limit_direction="both")

df[numeric_cols] = df[numeric_cols].ffill().bfill()

print("Any remaining NaN values:", int(df.isna().sum().sum()))


Any remaining NaN values: 0


## 6) Temporal Feature Engineering

This section builds time-aware predictors that help solar models capture daily and seasonal behavior:
- hour, month, day-of-year, weekday/weekend indicators
- holiday flag
- season one-hot features
- cyclical sine/cosine encoding for hour and day-of-year


In [9]:
# 4) Temporal feature creation (weekday/weekend/holiday + cyclic)
import importlib

# Dynamic import avoids static analyzer unresolved-import warnings in some setups
holiday_mod = importlib.import_module("pandas.tseries.holiday")
USFederalHolidayCalendar = holiday_mod.USFederalHolidayCalendar

# Build a datetime source that works even if index is not DatetimeIndex
if "DATE" in df.columns:
    dt = pd.to_datetime(df["DATE"], errors="coerce")
else:
    dt = pd.to_datetime(df.index, errors="coerce")

if dt.isna().any():
    valid = dt.notna()
    df = df.loc[valid].copy()
    dt = dt.loc[valid]

# Keep DATE synchronized as datetime for downstream steps
df["DATE"] = dt

# Optional: keep a datetime index for consistent time-series operations later
if not isinstance(df.index, pd.DatetimeIndex):
    df = df.set_index("DATE", drop=False)

# Calendar features
df["HOUR"] = dt.dt.hour.values

# Shift feature: 0-7, 8-15, 16-23
df["SHIFT"] = np.select(
    [
        df["HOUR"].between(0, 7),
        df["HOUR"].between(8, 15),
        df["HOUR"].between(16, 23)
    ],
    [1, 2, 3]
)

df["MONTH"] = dt.dt.month.values
df["DAY_OF_YEAR"] = dt.dt.dayofyear.values
df["DAY_OF_WEEK"] = dt.dt.dayofweek.values
df["IS_WEEKEND"] = (df["DAY_OF_WEEK"] >= 5).astype(int)
df["IS_WEEKDAY"] = (df["DAY_OF_WEEK"] < 5).astype(int)

# # Season encoding (Northern Hemisphere)
# season_map = {12: "winter", 1: "winter", 2: "winter", 3: "spring", 4: "spring", 5: "spring", 6: "summer", 7: "summer", 8: "summer", 9: "fall", 10: "fall", 11: "fall"}
# df["SEASON"] = df["MONTH"].map(season_map)
# df = pd.get_dummies(df, columns=["SEASON"], drop_first=False)

# Holiday indicator
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=dt.min().date(), end=dt.max().date())
df["IS_HOLIDAY"] = dt.dt.normalize().isin(holidays).astype(int).values

# Cyclic encodings
df["HOUR_SIN"] = np.sin(2 * np.pi * df["HOUR"] / 24)
df["HOUR_COS"] = np.cos(2 * np.pi * df["HOUR"] / 24)
df["DOY_SIN"] = np.sin(2 * np.pi * df["DAY_OF_YEAR"] / 365.25)
df["DOY_COS"] = np.cos(2 * np.pi * df["DAY_OF_YEAR"] / 365.25)



display(df.head())


,INDEX,DATE,TIME_INDEX,TCWL,TCIW,SP,HUM,TCC,U,V,TEMP,TP,SSRD,STRD,TSR,POWER,MISS_HUM,MISS_TEMP,MISS_SSRD,MISS_POWER,HOUR,SHIFT,MONTH,DAY_OF_YEAR,DAY_OF_WEEK,IS_WEEKEND,IS_WEEKDAY,IS_HOLIDAY,HOUR_SIN,HOUR_COS,DOY_SIN,DOY_COS
DATE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2012-04-01 00:00:00,0.0,2012-04-01 00:00:00,1.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.754103,0,0,0,0,0,1,4,92,6,1,0,0,0.000000,1.000000,0.99993,-0.011826
2012-04-01 01:00:00,1.0,2012-04-01 01:00:00,2.0,0.001967,0.003609,94843.6250,60.221909,0.244601,1.039334,-2.503039,294.448486,0.000000,3091744.5,1241430.5,3386228.5,0.555000,0,0,0,0,1,1,4,92,6,1,0,0,0.258819,0.965926,0.99993,-0.011826
2012-04-01 02:00:00,2.0,2012-04-01 02:00:00,3.0,0.005524,0.033575,94757.9375,54.678604,0.457138,2.482865,-2.993330,295.651367,0.000000,2671979.0,1239402.0,3038689.5,0.438397,0,0,0,0,2,1,4,92,6,1,0,0,0.500000,0.866025,0.99993,-0.011826
2012-04-01 03:00:00,3.0,2012-04-01 03:00:00,4.0,0.030113,0.132009,94732.8125,61.294891,0.771429,3.339867,-1.982535,294.454590,0.001341,2252213.5,1237373.5,2691150.5,0.145449,0,0,0,0,3,1,4,92,6,1,0,0,0.707107,0.707107,0.99993,-0.011826
2012-04-01 04:00:00,4.0,2012-04-01 04:00:00,5.0,0.057167,0.110645,94704.0625,67.775284,0.965866,3.106102,-1.446051,293.261475,0.002501,1610654.5,1286522.0,2083191.0,0.111987,0,0,0,0,4,1,4,92,6,1,0,0,0.866025,0.500000,0.99993,-0.011826


## 7) Feature Cleaning and Selection

Here, `POWER` is separated as the target, and numeric predictors are filtered to reduce noise:
- remove near-constant (low-variance) features
- remove highly correlated features (`> 0.98`) to reduce redundancy

This creates a cleaner feature matrix for downstream models.


In [10]:
# 5) Feature cleaning and selection
#remove missingness indicator features
df = df.loc[:, ~df.columns.str.startswith("MISS")]

target_col = "POWER"
if target_col not in df.columns:
    raise ValueError("POWER target column not found after preprocessing")

X = df.drop(columns=[target_col])
y = df[target_col].copy()

# Keep only numeric features for variance/correlation filtering
X_num = X.select_dtypes(include=[np.number]).copy()

# Low-variance removal
vt = VarianceThreshold(threshold=1e-8)
X_vt = pd.DataFrame(vt.fit_transform(X_num), index=X_num.index, columns=X_num.columns[vt.get_support()])
removed_low_variance = sorted(set(X_num.columns) - set(X_vt.columns))
print("Low-variance features removed:", len(removed_low_variance))
print("Low-variance feature list:", removed_low_variance)

# High-correlation removal
corr = X_vt.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_cols = [c for c in upper.columns if (upper[c] > 0.98).any()]
X_final = X_vt.drop(columns=high_corr_cols)
print("Highly correlated features removed:", len(high_corr_cols))
print("Highly correlated feature list:", high_corr_cols)
print("Final feature shape:", X_final.shape)


Low-variance features removed: 0
Low-variance feature list: []
Highly correlated features removed: 3
Highly correlated feature list: ['TSR', 'DAY_OF_YEAR', 'IS_WEEKDAY']
Final feature shape: (19704, 23)


In [11]:
# High-correlation removal
corr = X_vt.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

threshold = 0.98

# Find highly correlated feature pairs
high_corr_pairs = []
for col in upper.columns:
    correlated_features = upper.index[upper[col] > threshold].tolist()
    for row in correlated_features:
        high_corr_pairs.append((row, col, upper.loc[row, col]))

# Print correlated pairs
print(f"\nHighly correlated feature pairs (>|{threshold}|):")
for f1, f2, corr_val in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True):
    print(f"{f1:30s} <--> {f2:30s} : {corr_val:.4f}")

# Features to remove
high_corr_cols = [c for c in upper.columns if (upper[c] > threshold).any()]

X_final = X_vt.drop(columns=high_corr_cols)

print("\nHighly correlated features removed:", len(high_corr_cols))
print("Highly correlated feature list:", high_corr_cols)
print("Final feature shape:", X_final.shape)



Highly correlated feature pairs (>|0.98|):
IS_WEEKEND                     <--> IS_WEEKDAY                     : 1.0000
MONTH                          <--> DAY_OF_YEAR                    : 0.9962
SSRD                           <--> TSR                            : 0.9931

Highly correlated features removed: 3
Highly correlated feature list: ['TSR', 'DAY_OF_YEAR', 'IS_WEEKDAY']
Final feature shape: (19704, 23)


In [114]:
vt

VarianceThreshold(threshold=1e-08)

## 8) Export `X_final` and Data Quality Report

This section removes train/validation/test scaling and instead:
- checks NaN values in `X_final` and `y`
- prints a concise data quality report
- saves `X_final` directly to CSV for downstream modeling


In [12]:
# 6) NaN check + report + save X_final
# Ensure target has a stable column name for combined export
y_named = y.rename("POWER")
combined_final = pd.concat([X_final, y_named], axis=1)

after_selection_report = {
    "X_final_rows": int(X_final.shape[0]),
    "X_final_cols": int(X_final.shape[1]),
    "y_rows": int(y_named.shape[0]),
    "combined_rows": int(combined_final.shape[0]),
    "combined_cols": int(combined_final.shape[1]),
    "X_final_total_nan": int(X_final.isna().sum().sum()),
    "y_total_nan": int(y_named.isna().sum()),
    "combined_total_nan": int(combined_final.isna().sum().sum()),
}

x_nan_by_col = X_final.isna().sum()
x_nan_by_col = x_nan_by_col[x_nan_by_col > 0].sort_values(ascending=False)

print("Data quality report:")
print(after_selection_report)
print("\nColumns with NaN in X_final:")
print(x_nan_by_col if not x_nan_by_col.empty else "None")

out_dir = Path("data") / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

X_final.to_csv(out_dir / "X_final.csv")
y_named.to_csv(out_dir / "y_final.csv", header=True)
combined_final.to_csv(out_dir / "combined_final.csv")

print(f"\nSaved X_final to: {(out_dir / 'X_final.csv').resolve()}")
print(f"Saved y to: {(out_dir / 'y_final.csv').resolve()}")
print(f"Saved combined_final to: {(out_dir / 'combined_final.csv').resolve()}")


Data quality report:
{'X_final_rows': 19704, 'X_final_cols': 23, 'y_rows': 19704, 'combined_rows': 19704, 'combined_cols': 24, 'X_final_total_nan': 0, 'y_total_nan': 0, 'combined_total_nan': 0}

Columns with NaN in X_final:
None

Saved X_final to: /Users/mkuzlu/Dropbox/Technical_Projects/2026_MKS_TimeSer_SolarFor/Source_Code/data/processed/X_final.csv
Saved y to: /Users/mkuzlu/Dropbox/Technical_Projects/2026_MKS_TimeSer_SolarFor/Source_Code/data/processed/y_final.csv
Saved combined_final to: /Users/mkuzlu/Dropbox/Technical_Projects/2026_MKS_TimeSer_SolarFor/Source_Code/data/processed/combined_final.csv


## 9) Quick Verification of Saved Files

This optional check confirms the saved files exist and shows their shapes after reloading from disk.


In [13]:
# Optional: reload saved files for verification
x_path = Path("data") / "processed" / "X_final.csv"
y_path = Path("data") / "processed" / "y_final.csv"
combined_path = Path("data") / "processed" / "combined_final.csv"

X_final_reloaded = pd.read_csv(x_path, index_col=0)
y_reloaded = pd.read_csv(y_path, index_col=0)
combined_reloaded = pd.read_csv(combined_path, index_col=0)

print("Reloaded X_final shape:", X_final_reloaded.shape)
print("Reloaded y shape:", y_reloaded.shape)
print("Reloaded combined shape:", combined_reloaded.shape)
print("Reloaded X_final total NaN:", int(X_final_reloaded.isna().sum().sum()))
print("Reloaded y total NaN:", int(y_reloaded.isna().sum().sum()))
print("Reloaded combined total NaN:", int(combined_reloaded.isna().sum().sum()))


Reloaded X_final shape: (19704, 23)
Reloaded y shape: (19704, 1)
Reloaded combined shape: (19704, 24)
Reloaded X_final total NaN: 0
Reloaded y total NaN: 0
Reloaded combined total NaN: 0
